# Land-Cover Classification

Use the launch button at the top of this page to open the full notebook in Google Colab. This workflow trains and evaluates land-cover classifiers in Earth Engine.


In [ ]:
import ee
import geemap

In [ ]:
ee.Authenticate()

In [ ]:
ee.Initialize(project='your-cloud-project')

# **Case 1**

## Task 1: Choose a target Landsat image

In [ ]:
# Define a function that scales and masks Landsat 8 surface reflectance images.
def prep_sr_l8(image):
  """Scales and masks Landsat 8 surface reflectance images."""
  # Develop masks for unwanted pixels (fill, cloud, cloud shadow).
  qa_mask = image.select('QA_PIXEL').bitwiseAnd(0b11111).eq(0)
  saturation_mask = image.select('QA_RADSAT').eq(0)

  # Apply the scaling factors to the appropriate bands.
  def _get_factor_img(factor_names):
    factor_list = image.toDictionary().select(factor_names).values()
    return ee.Image.constant(factor_list)

  scale_img = _get_factor_img([
      'REFLECTANCE_MULT_BAND_.|TEMPERATURE_MULT_BAND_ST_B10'])
  offset_img = _get_factor_img([
      'REFLECTANCE_ADD_BAND_.|TEMPERATURE_ADD_BAND_ST_B10'])
  scaled = image.select('SR_B.|ST_B10').multiply(scale_img).add(offset_img)

  # Replace original bands with scaled bands and apply masks.
  return image.addBands(scaled, None, True).updateMask(
      qa_mask).updateMask(saturation_mask)

In [ ]:
l8_image_1 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2021-03-01', '2021-07-01').map(prep_sr_l8).median()

In [ ]:
l8_image_1

In [ ]:
# Use these bands for prediction.
selected_bands_1 = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'ST_B10']

## Task 2: Load labelled training points

In [ ]:
# Load the labelled demo point set from the Earth Engine data catalog.
labeled_points = ee.FeatureCollection('GOOGLE/EE/DEMOS/demo_landcover_labels')

In [ ]:
# This property stores the land cover labels as consecutive integers starting from zero.
label = 'landcover'

In [ ]:
labeled_points

In [ ]:
Map = geemap.Map()

In [ ]:
Map.add_basemap("Esri.WorldImagery")

In [ ]:
Map.addLayer(labeled_points, {'color':'red'}, 'sample points')

In [ ]:
Map.centerObject(labeled_points, zoom=10)

In [ ]:
Map

## Task 3: Sample the image and split training/test sets

In [ ]:
# Extract training samples from the image.
samples = l8_image_1.select(selected_bands_1).sampleRegions(
    collection=labeled_points,  # Labelled points to sample from the image.
    properties=[label],         # Keep the label property, such as landcover.
    scale=30                    # Sampling resolution in metres; Landsat optical bands are 30 m.
)

In [ ]:
samples.aggregate_stats('landcover').getInfo()

In [ ]:
samples.aggregate_histogram('landcover').getInfo()

In [ ]:
# Add a random number column for splitting training and validation data.
samples = samples.randomColumn('random')      # Uniform random values in [0, 1).
# samples = samples.randomColumn('random', seed=42)

In [ ]:
# Split into training and test sets with a 70/30 threshold.
train_set = samples.filter(ee.Filter.lt('random', 0.7))
test_set = samples.filter(ee.Filter.gte('random', 0.7))

In [ ]:
train_set.aggregate_histogram('landcover').getInfo()

In [ ]:
test_set.aggregate_histogram('landcover').getInfo()

In [ ]:
train_set

## Task 4: Train the classifier

In [ ]:
classifier_cart = ee.Classifier.smileCart()

In [ ]:
trained_classifier_cart = classifier_cart.train(
    features=train_set,
    classProperty=label,
    inputProperties=selected_bands_1
)

## Task 5: Evaluate accuracy on the test set

In [ ]:
classified_test_set = test_set.classify(trained_classifier_cart)

In [ ]:
# Compute the confusion matrix.
conf_matrix_1 = classified_test_set.errorMatrix(label, 'classification')

In [ ]:
print('Confusion matrix:\n', conf_matrix_1.getInfo())
print('Overall Accuracy:', conf_matrix_1.accuracy().getInfo())
print('Kappa:', conf_matrix_1.kappa().getInfo())

## Task 6: Classify the image and display it with geemap

In [ ]:
# Classify the image with the same bands used for training.
classified_l8_image_1 = l8_image_1.select(selected_bands_1).classify(trained_classifier_cart)

In [ ]:
m = geemap.Map()
m.set_center(-122.0877, 37.7880, 11)

In [ ]:
# Set style parameters.
vis_params = {
    'color': 'red',
    'pointSize': 5,
    'pointShape': 'circle'  # Options include circle, square, and triangle.
}

In [ ]:
# # Add the original image.
# m.add_layer(
#     l8_image_1,
#     {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0, 'max': 0.25},
#     'image',
# )

In [ ]:
# Add the classified image.
m.add_layer(
    classified_l8_image_1,
    {'min': 0, 'max': 2, 'palette': ['orange', 'green', 'blue']},
    'classification',
)

In [ ]:
m

## Bonus: Visualize the decision tree with scikit-learn

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
import matplotlib.pyplot as plt

In [ ]:
# Assume samples is already an Earth Engine FeatureCollection.
# Extract all features and convert them to Python dictionaries.
sample_list = samples.getInfo()['features']

# Convert the feature properties to a DataFrame.
rows = []
for feature in sample_list:
    row = feature['properties']
    rows.append(row)

df = pd.DataFrame(rows)

In [ ]:
# Assume the label column is named 'landcover'.
X = df.drop(columns=['landcover'])  # Features
y = df['landcover']                # Labels

In [ ]:
clf = DecisionTreeClassifier(max_depth=4, random_state=0)
clf.fit(X, y)

In [ ]:
r = tree.export_text(clf, feature_names=list(X.columns))
print(r)

In [ ]:
plt.figure(figsize=(20,10))
tree.plot_tree(clf,
               feature_names=X.columns,
               class_names=[str(cls) for cls in clf.classes_],
               filled=True,
               rounded=True,
               fontsize=12)
plt.show()

# **Case 2**

In [ ]:
bbox_1 = ee.Geometry.BBox(-86.132813, -30.448674, -28.652344, 14.604847)

# Make a cloud-free Landsat 8 surface reflectance composite.
l8_image_2 = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterDate('2018-01-01', '2019-01-01').filterBounds(bbox_1)
    .map(prep_sr_l8)
    .median())

In [ ]:
# Use these bands for prediction.
selected_bands_2 = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']

In [ ]:
# Define custom sampling regions.
forest1 = ee.Geometry.Rectangle(-63.0187, -9.3958, -62.9793, -9.3443)
forest2 = ee.Geometry.Rectangle(-62.8145, -9.206, -62.7688, -9.1735)
non_forest1 = ee.Geometry.Rectangle(-62.8161, -9.5001, -62.7921, -9.4486)
non_forest2 = ee.Geometry.Rectangle(-62.6788, -9.044, -62.6459, -8.9986)

In [ ]:
# Combine the sampling regions into a FeatureCollection.
regions_to_sample = ee.FeatureCollection([
    ee.Feature(non_forest1, {'class': 0}),        # Use class labels to mark forest vs non-forest.
    ee.Feature(non_forest2, {'class': 0}),
    ee.Feature(forest1, {'class': 1}),
    ee.Feature(forest2, {'class': 1}),
])

In [ ]:
m = geemap.Map(center=[-9.2399, -62.836], zoom=10, basemap='Esri.WorldImagery')

In [ ]:
m

In [ ]:
m.add_layer(regions_to_sample, {'color': 'yellow'}, 'sampled polygons')

In [ ]:
# Get the values for all pixels in each training polygon.
samples_2 = l8_image_2.sampleRegions(
    collection=regions_to_sample,   # Regions sampled from the larger Landsat 8 image.
    properties=['class'],           # Keep the custom class label.
    scale=30,                       # Sampling resolution in metres; Landsat optical bands are 30 m.
)

In [ ]:
classifier_2 = ee.Classifier.smileCart(
    maxNodes=10,
    minLeafPopulation=5
)

In [ ]:
# Train the classifier.
trained_classifier_2 = classifier_2.train(samples_2, 'class', selected_bands_2)

In [ ]:
# Classify the image.
classified_l8_image_2 = l8_image_2.classify(trained_classifier_2)

In [ ]:
m.add_layer(
    classified_l8_image_2,
    {'min': 0, 'max': 1,
     'palette': [
         'orange',   # non-forest
         'green'     # forest
         ]},
    'deforestation',
)

In [ ]:
m

# **Case 3**

In [ ]:
# Define a region of interest.
roi = ee.Geometry.BBox(-122.93, 36.99, -121.20, 38.16)

In [ ]:
# Make a cloud-free Landsat 8 surface reflectance composite.
landsat_image_1 = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(roi)
    .filterDate('2019-03-01', '2019-07-01')
    .map(prep_sr_l8)
    .median()
    .setDefaultProjection('EPSG:4326', None, 30)
    .select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'])
    .clip(roi)
)

In [ ]:
# Use MODIS land cover, IGBP classification, for training.
modis_image_1 = ee.Image('MODIS/061/MCD12Q1/2020_01_01').select('LC_Type1').clip(roi)

In [ ]:
# Define a palette for the IGBP classification.
igbp_palette = [
    'aec3d4',  # water
    '152106', '225129', '369b47', '30eb5b', '387242',  # forest
    '6a2325', 'c3aa69', 'b76031', 'd9903d', '91af40',  # shrub, grass
    '111149',  # wetlands
    'cdb33b',  # croplands
    'cc0013',  # urban
    '33280d',  # crop mosaic
    'd7cdcc',  # snow and ice
    'f7e084',  # barren
    '6f6f6f'   # tundra
]

In [ ]:
Map = geemap.Map(basemap='Esri.WorldImagery')
Map.addLayer(roi, {}, 'roi')
Map.addLayer(landsat_image_1, {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min':0.0, 'max': 0.30}, 'Landsat Image 1')
Map.addLayer(modis_image_1, {'palette': igbp_palette, 'min': 0, 'max': 17}, 'Modis Image 1')
Map.center_object(roi, 9)

In [ ]:
Map

**MODIS MCD12Q1 land-cover product overview**

https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MCD12Q1

| Property | Description |
| -------- | ----------- |
| **Product** | MCD12Q1, MODIS Land Cover Type Yearly L3 Global 500 m |
| **Resolution** | 500 m spatial resolution |
| **Time span** | Annual products from 2001 onward |
| **Temporal interval** | One image per year |
| **Projection** | Sinusoidal projection |
| **Data source** | Joint Terra and Aqua MODIS observations |

Band overview

| Band name | Description |
| --------- | ----------- |
| **LC_Type1** | IGBP classification, the most widely used land-cover type scheme |
| LC_Type2 | UMD classification |
| LC_Type3 | MODIS-derived LAI/FPAR classification scheme |
| LC_Type4 | NPP classification |
| LC_Type5 | Plant Functional Type classification |

In [ ]:
# Sample the image to obtain training data.
training_points = landsat_image_1.addBands(modis_image_1).sample(
    region=roi,            # Limit sampling to the ROI.
    numPixels=5000,        # Randomly sample up to 5,000 pixels.
    seed=0                 # Set a random seed for reproducibility.
)

In [ ]:
Map.addLayer(training_points, {'color': 'red'}, 'Sample Points')

In [ ]:
Map

In [ ]:
# Define and train a Random Forest classifier.
trained_RM_classifier = ee.Classifier.smileRandomForest(100).train(
    features=training_points,
    classProperty='LC_Type1',
    inputProperties=['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'],
)

In [ ]:
# ee.Classifier.smileRandomForest(numberOfTrees, variablesPerSpilt, minLeafPopulation, bagFraction, maxNodes, seed)

In [ ]:
# Classify every pixel in the Landsat image.
classified_landsat_image_1 = landsat_image_1.classify(trained_RM_classifier)

In [ ]:
# Generate a resubstitution confusion matrix to evaluate the model fit on the training data.
resubtitution_confusion_matrix = trained_RM_classifier.confusionMatrix()

In [ ]:
resubtitution_confusion_matrix

In [ ]:
resubtitution_confusion_matrix.getInfo()

In [ ]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

In [ ]:
df_resubtitution_confusion_matrix = pd.DataFrame(resubtitution_confusion_matrix.getInfo())

In [ ]:
igbp_labels = [
    'Evergreen Needleleaf', 'Evergreen Broadleaf', 'Deciduous Needleleaf',
    'Deciduous Broadleaf', 'Mixed Forest', 'Closed Shrubland',
    'Open Shrubland', 'Woody Savannas', 'Savannas', 'Grasslands',
    'Permanent Wetlands', 'Croplands', 'Urban', 'Crop/Natural Mosaic',
    'Snow/Ice', 'Barren', 'Water', 'Tundra'
]

df_resubtitution_confusion_matrix.index = igbp_labels
df_resubtitution_confusion_matrix.columns = igbp_labels

In [ ]:
fig = plt.figure(figsize=(10, 10), constrained_layout=True)

ax1 = plt.subplot(111)
sns.heatmap(df_resubtitution_confusion_matrix, annot=True, fmt='d', cmap='Reds', ax=ax1)
plt.xlabel('predicted')
plt.ylabel('actual')

for i in range(len(df_resubtitution_confusion_matrix)):
    ax1.add_patch(plt.Rectangle((i, i), 1, 1, fill=False, edgecolor='grey', lw=1))

plt.show()

In [ ]:
resubtitution_confusion_matrix.accuracy()

In [ ]:
resubtitution_confusion_matrix.consumersAccuracy()     # user accuracy

In [ ]:
resubtitution_confusion_matrix.producersAccuracy()    # producer accuracy

In [ ]:
import numpy as np

In [ ]:
train_assessment = pd.DataFrame([], index=igbp_labels)
train_assessment['accuracy'] = np.array(resubtitution_confusion_matrix.consumersAccuracy().getInfo()).squeeze()
train_assessment['recall'] = np.array(resubtitution_confusion_matrix.producersAccuracy().getInfo()).squeeze()

In [ ]:
fig = plt.figure(figsize=(10, 3))
ax1 = plt.subplot(111)
train_assessment.plot.bar(ax=ax1)
plt.show()

In [ ]:
print('Training overall accuracy:', resubtitution_confusion_matrix.accuracy().getInfo())

**Independent-sample validation:** sample the original Landsat image again to test how well the model generalizes.

In [ ]:
validation_points_1 = landsat_image_1.addBands(modis_image_1).sample(
    region=roi,
    numPixels=5000,
    seed=1,      # Use a different random seed.
    ).filter(ee.Filter.notNull(landsat_image_1.bandNames()))    # Filter the result to get rid of any null pixels.

In [ ]:
# Classify the validation points with the trained classifier.
classified_validation_points_1 = validation_points_1.classify(trained_RM_classifier)

In [ ]:
# Build a confusion matrix to test model generalization.
confusion_matrix_1 = classified_validation_points_1.errorMatrix('LC_Type1', 'classification')

In [ ]:
validation_1 = pd.DataFrame([], index=igbp_labels)
validation_1['accuracy'] = np.array(confusion_matrix_1.consumersAccuracy().getInfo()).squeeze()
validation_1['recall'] = np.array(confusion_matrix_1.producersAccuracy().getInfo()).squeeze()

In [ ]:
fig = plt.figure(figsize=(10, 3))
ax1 = plt.subplot(111)
validation_1.plot.bar(ax=ax1)
plt.show()

In [ ]:
df_validation_matrix = pd.DataFrame(confusion_matrix_1.getInfo())
df_validation_matrix.index = igbp_labels
df_validation_matrix.columns = igbp_labels

In [ ]:
fig = plt.figure(figsize=(10, 10), constrained_layout=True)

ax1 = plt.subplot(111)
sns.heatmap(df_validation_matrix, annot=True, fmt='d', cmap='YlGnBu', ax=ax1)
plt.xlabel('predicted')
plt.ylabel('actual')

for i in range(len(df_validation_matrix)):
    ax1.add_patch(plt.Rectangle((i, i), 1, 1, fill=False, edgecolor='grey', lw=1))

plt.show()

In [ ]:
confusion_matrix_1.accuracy()

In [ ]:
Map

In [ ]:
other_region = Map.user_roi

In [ ]:
other_region

In [ ]:
landsat_image_2 = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(other_region)
    .filterDate('2020-03-01', '2020-07-01')
    .map(prep_sr_l8)
    .median()
    .setDefaultProjection('EPSG:4326', None, 30)
    .select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']).clip(other_region)
)

In [ ]:
modis_image_2 = ee.Image('MODIS/061/MCD12Q1/2020_01_01').select('LC_Type1').clip(other_region)

In [ ]:
validation_points_2 = landsat_image_2.addBands(modis_image_2).sample(
    region=other_region,
    numPixels=5000,
    seed=1,      # Use a different random seed.
    ).filter(ee.Filter.notNull(landsat_image_2.bandNames()))    # Filter the result to get rid of any null pixels.

In [ ]:
# Classify the validation points with the trained classifier.
classified_validation_points_2 = validation_points_2.classify(trained_RM_classifier)

In [ ]:
# Build a confusion matrix to test model generalization.
confusion_matrix_2 = classified_validation_points_2.errorMatrix('LC_Type1', 'classification')

In [ ]:
validation_2 = pd.DataFrame([], index=igbp_labels)
validation_2['accuracy'] = np.array(confusion_matrix_2.consumersAccuracy().getInfo()).squeeze()
validation_2['recall'] = np.array(confusion_matrix_2.producersAccuracy().getInfo()).squeeze()

In [ ]:
fig = plt.figure(figsize=(10, 3))
ax1 = plt.subplot(111)
validation_2.plot.bar(ax=ax1)
plt.show()

In [ ]:
display('Validation overall accuracy:', confusion_matrix_2.accuracy())

In [ ]:
df_validation_matrix_2 = pd.DataFrame(confusion_matrix_2.getInfo())
df_validation_matrix_2.index = igbp_labels
df_validation_matrix_2.columns = igbp_labels

In [ ]:
fig = plt.figure(figsize=(10, 10), constrained_layout=True)

ax1 = plt.subplot(111)
sns.heatmap(df_validation_matrix_2, annot=True, fmt='d', cmap='YlGnBu', ax=ax1)
plt.xlabel('predicted')
plt.ylabel('actual')

for i in range(len(df_validation_matrix_2)):
    ax1.add_patch(plt.Rectangle((i, i), 1, 1, fill=False, edgecolor='grey', lw=1))

plt.show()

In [ ]:
# Classify the validation Landsat image.
classified_landsat_image_2 = landsat_image_2.classify(trained_RM_classifier)

In [ ]:
Map = geemap.Map(basemap='Esri.WorldImagery')
roi_and_other_region = ee.FeatureCollection([roi, other_region])
Map.addLayer(roi_and_other_region, {}, 'roi')
Map.center_object(roi_and_other_region, 8)

In [ ]:
# Display the input and the classification with geemap in a notebook.
Map.add_layer(
    ee.ImageCollection([landsat_image_1, landsat_image_2]),
    {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0, 'max': 0.25},
    'landsat',
)

In [ ]:
Map.add_layer(
    ee.ImageCollection([classified_landsat_image_1, classified_landsat_image_2]),
    {'palette': igbp_palette, 'min': 0, 'max': 17},
    'classification',
)

In [ ]:
Map.add_layer(
    ee.ImageCollection([modis_image_1, modis_image_2]),
    {'palette': igbp_palette, 'min': 0, 'max': 17},
    'MODIS LC',
)

In [ ]:
Map

# Task 7: Save the trained model

In [ ]:
# Using the random forest classifier defined earlier, export the random
# forest classifier as an Earth Engine asset.
classifier_asset_id = (
    'projects/<PROJECT-ID>/assets/upscaled_MCD12Q1_random_forest'
)
task = ee.batch.Export.classifier.toAsset(
    trained_RM_classifier, 'Saved-random-forest-IGBP-classification', classifier_asset_id
)

In [ ]:
task.start()

# Task 8: Load the saved model, classify an image, and display the result

In [ ]:
# Once the classifier export finishes, we can load our saved classifier.
saved_classifier = ee.Classifier.load(classifier_asset_id)
# We can perform classification just as before with the saved classifier now.
# other_image =
classified = other_image.classify(saved_classifier)

In [ ]:
m = geemap.Map()
m.center_object(roi, 10)
m.add_layer(
    classified.clip(roi),
    {'palette': igbp_palette, 'min': 0, 'max': 17},
    'classification',
)

## References

- Google Earth Engine Developers. (n.d.). [Supervised Classification](https://developers.google.com/earth-engine/guides/classification).
- Google Earth Engine API Reference. (n.d.). [`ee.Classifier.smileRandomForest`](https://developers.google.com/earth-engine/apidocs/ee-classifier-smilerandomforest).
- Google Earth Engine Data Catalog. (n.d.). [USGS Landsat 8 Level 2, Collection 2, Tier 1](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2).
- Google Earth Engine Data Catalog. (n.d.). [MCD12Q1.061 MODIS Land Cover Type Yearly Global 500m](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MCD12Q1).
- Google Earth Engine demo dataset. (n.d.). `GOOGLE/EE/DEMOS/demo_landcover_labels`, used as example training labels in the Earth Engine supervised-classification workflow.
